In [ ]:
!pip install llama-index
!pip install llama-index-core==0.10.42
!pip install llama-index-embeddings-openai
!pip install llama-index-postprocessor-flag-embedding-reranker
!pip install git+https://github.com/FlagOpen/FlagEmbedding.git
!pip install llama-index-graph-stores-neo4j
!pip install llama-cloud-services

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.9/253.9 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.3/302.3 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 50.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.2/129.2 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.4/15.4 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.9/141.9 kB 13.2 MB/s eta 0:00:00
  Attempting uninstall: tenacity
    Found existing installation: tenacity 9.1.2
    Uninstalling tenacity-9.1.2:
      Successfully uninstalled tenacity-9.1.2
  Attempting uninstall: llama-index-core
    Found existing installation: llama-index-core 0.12.28
    Uninstalling llama-index

In [ ]:
import nest_asyncio

nest_asyncio.apply()

In [ ]:
import os
from google.colab import userdata

os.environ["LLAMA_CLOUD_API_KEY"] =  userdata.get('LLAMA_CLOUD_API_KEY')
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

#### Setup Model

In [ ]:
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding
from llama_index.core import Settings

llm = OpenAI(model="gpt-4o")
embed_model = OpenAIEmbedding(model="text-embedding-3-small")

Settings.llm = llm
Settings.embed_model = embed_model

#### Load Data

In [ ]:
!mkdir data
!wget "https://www.dropbox.com/scl/fi/vip161t63s56vd94neqlt/2023-CSF_Proposed_Budget_Book_June_2023_Master_Web.pdf?rlkey=hemoce3w1jsuf6s2bz87g549i&dl=0" -O data/budget_2023.pdf

mkdir: cannot create directory ‘data’: File exists
--2025-04-06 02:09:48--  https://www.dropbox.com/scl/fi/vip161t63s56vd94neqlt/2023-CSF_Proposed_Budget_Book_June_2023_Master_Web.pdf?rlkey=hemoce3w1jsuf6s2bz87g549i&dl=0
Resolving www.dropbox.com (www.dropbox.com)... 162.125.67.18, 2620:100:6025:18::a27d:4512
Connecting to www.dropbox.com (www.dropbox.com)|162.125.67.18|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://uc0ccffa73be1e3a645351e8261c.dl.dropboxusercontent.com/cd/0/inline/CnQXHVJzmHc7eeVZyvMgq0JVgynQ7RORjSL4vt-9iPd5hXYecbZIyQvlcxem_lGaJEc0ph9mwTRK1LicOYA19p5Wh0D9mtDY76dsD7ys5LOiFGfqSWu_XQWTjUfF-xSTfSk/file# [following]
--2025-04-06 02:09:49--  https://uc0ccffa73be1e3a645351e8261c.dl.dropboxusercontent.com/cd/0/inline/CnQXHVJzmHc7eeVZyvMgq0JVgynQ7RORjSL4vt-9iPd5hXYecbZIyQvlcxem_lGaJEc0ph9mwTRK1LicOYA19p5Wh0D9mtDY76dsD7ys5LOiFGfqSWu_XQWTjUfF-xSTfSk/file
Resolving uc0ccffa73be1e3a645351e8261c.dl.dropboxusercontent.com (uc0ccffa73be1e3a6453

In [ ]:
from llama_cloud_services import LlamaParse

docs = LlamaParse(result_type="text").load_data("./data/budget_2023.pdf")

Started parsing the file under job_id 47c1a128-d00c-49d1-afe5-934bb1150f6d


In [ ]:
from copy import deepcopy
from llama_index.core.schema import TextNode, Document
from llama_index.core import VectorStoreIndex


def get_sub_docs(docs):
    """Split docs into pages, by separator."""
    sub_docs = []
    for doc in docs:
        doc_chunks = doc.text.split("\n---\n")
        for doc_chunk in doc_chunks:
            sub_doc = Document(
                text=doc_chunk,
                metadata=deepcopy(doc.metadata),
            )
            sub_docs.append(sub_doc)

    return sub_docs

In [ ]:
# this will split into pages
sub_docs = get_sub_docs(docs)

#### Initialize Graph Store

To launch Neo4j locally, first ensure you have docker installed. Then, you can launch the database with the following docker command

```bash
docker run \
    -p 7474:7474 -p 7687:7687 \
    -v $PWD/data:/data -v $PWD/plugins:/plugins \
    --name neo4j-apoc \
    -e NEO4J_apoc_export_file_enabled=true \
    -e NEO4J_apoc_import_file_enabled=true \
    -e NEO4J_apoc_import_file_use__neo4j__config=true \
    -e NEO4JLABS_PLUGINS=\[\"apoc\"\] \
    neo4j:latest
```

From here, you can open the db at [http://localhost:7474/](http://localhost:7474/). On this page, you will be asked to sign in. Use the default username/password of `neo4j` and `neo4j`.

Once you login for the first time, you will be asked to change the password.

After this, you are ready to create your first property graph!

In [ ]:
from llama_index.graph_stores.neo4j import Neo4jPGStore

graph_store = Neo4jPGStore(
    username="neo4j",
    password="llamaindex",
    url="bolt://localhost:7687",
)
vec_store = None

## Construct Knowledge Graph, Get Retrievers

In [ ]:
from llama_index.core.indices.property_graph import (
    ImplicitPathExtractor,
    SimpleLLMPathExtractor,
)
from llama_index.core import PropertyGraphIndex
from llama_index.llms.openai import OpenAI
from llama_index.embeddings.openai import OpenAIEmbedding

In [ ]:
index = PropertyGraphIndex.from_documents(
    sub_docs,
    embed_model=OpenAIEmbedding(model_name="text-embedding-3-small"),
    kg_extractors=[
        ImplicitPathExtractor(),
        SimpleLLMPathExtractor(
            llm=OpenAI(model="gpt-3.5-turbo", temperature=0.3),
            num_workers=4,
            max_paths_per_chunk=10,
        ),
    ],
    property_graph_store=graph_store,
    show_progress=True,
)

NameError: name 'PropertyGraphIndex' is not defined

In [ ]:
# run this if index is already loaded
index = PropertyGraphIndex.from_existing(
    graph_store,
    embed_model=OpenAIEmbedding(model_name="text-embedding-3-small"),
    kg_extractors=[
        ImplicitPathExtractor(),
        SimpleLLMPathExtractor(
            llm=OpenAI(model="gpt-3.5-turbo", temperature=0.3),
            num_workers=4,
            max_paths_per_chunk=10,
        ),
    ],
    show_progress=True,
)

#### Define Vector Retriever

In [ ]:
from llama_index.core.indices.property_graph import VectorContextRetriever

kg_retriever = VectorContextRetriever(
    index.property_graph_store,
    embed_model=OpenAIEmbedding(model_name="text-embedding-3-small"),
    similarity_top_k=2,
    path_depth=1,
    # include_text=False,
    include_text=True,
)

In [ ]:
nodes = kg_retriever.retrieve(
    "Give me all the programs that the mayor's budget includes"
)
# nodes = kg_retriever.retrieve('san francisco')
print(len(nodes))
for idx, node in enumerate(nodes):
    print(f">> IDX: {idx}, {node.get_content()}")

3
>> IDX: 0, Here are some facts extracted from the provided text:

Mayor's proposed budget -> Balanced with -> Combination of departmental budget reductions
Mayor's proposed budget -> Contains -> Citywide budgetary and fiscal policy information
Mayor's proposed budget -> Enhances -> Investments
Mayor's proposed budget -> Organized into -> Sections
Mayor's proposed budget -> Makes -> Significant investment
Mayor's proposed budget -> Includes -> $0.7 million in fy 2023-24
Mayor's proposed budget -> Includes -> $0.6 million in fy 2024-25
Mayor's proposed budget -> Allocates -> $36.0 million
Mayor's proposed budget -> Includes funding for -> Legislative management system
Mayor's proposed budget -> Includes -> $2.6 million in fy 2024-25
Mayor's proposed budget -> Maintains -> Dcyf's technical assistance programming budget
Mayor's proposed budget -> Includes -> $2.5 million in fy 2023-24
Mayor's proposed budget -> Includes -> Anticipated grant
Mayor's proposed budget -> Includes -> Signific

## Build Baseline Vector Index

In [ ]:
from llama_index.core import VectorStoreIndex
from llama_index.core.query_engine import RetrieverQueryEngine

base_index = VectorStoreIndex.from_documents(sub_docs, embed_model=embed_model)
base_retriever = base_index.as_retriever(similarity_top_k=2)
base_query_engine = RetrieverQueryEngine(base_retriever)

In [ ]:
response = base_query_engine.query(
    "Give me all the programs that the mayor's budget includes"
)
print(str(response))

The mayor's budget includes programs such as financial capability services, nonprofit capacity building, eviction prevention and housing stabilization services, community and housing place-based services, civil legal services, supportive housing for persons with HIV/AIDS, community, coalition and cultural district building, rental and homeownership counseling, capital projects, and housing development grants. Additionally, there is a focus on creating permanently affordable housing, fostering healthy communities and neighborhoods, improving access to affordable housing, preserving affordable housing, and promoting self-sufficiency and protecting rights.


In [ ]:
print(len(response.source_nodes))
for node in response.source_nodes:
    print("---")
    print(node.get_content())

2
---
ORGANIZATIONAL STRUCTURE: MAYOR
                                                                                 Mayor
     Communications                         Administration                         Legislative &                        Public Policy
                                                                               Government Affairs                         & Finance
                                                                                                Housing & Community
                                                                                                     Development
Department Total Budget Historical Comparison (Mayor's Proposed)                                                  Budget Year 2023-2024 and 2024-2025
                                                           Department Total Budget Historical Comparison
                   TOTAL BUDGET – HISTORICAL COMPARISON
MYR Mayor                                                  2022-2023 

## Build Custom Retriever

In [ ]:
from llama_index.core.retrievers import BaseRetriever
from llama_index.core.schema import NodeWithScore
from typing import List


class CustomRetriever(BaseRetriever):
    """Custom retriever that performs both KG vector search and direct vector search."""

    def __init__(self, kg_retriever, vector_retriever):
        self._kg_retriever = kg_retriever
        self._vector_retriever = vector_retriever

    def _retrieve(self, query_bundle) -> List[NodeWithScore]:
        """Retrieve nodes given query."""
        kg_nodes = self._kg_retriever.retrieve(query_bundle)
        vector_nodes = self._vector_retriever.retrieve(query_bundle)

        unique_nodes = {n.node_id: n for n in kg_nodes}
        unique_nodes.update({n.node_id: n for n in vector_nodes})
        return list(unique_nodes.values())

In [ ]:
custom_retriever = CustomRetriever(kg_retriever, base_retriever)

In [ ]:
nodes = custom_retriever.retrieve(
    "Give me all the programs that the mayor's budget includes"
)
# len(nodes)

## Build Agent

In [ ]:
from llama_index.core.tools import QueryEngineTool, ToolMetadata
from llama_index.core.query_engine import RetrieverQueryEngine

kg_query_engine = RetrieverQueryEngine(custom_retriever)
kg_query_tool = QueryEngineTool(
    query_engine=kg_query_engine,
    metadata=ToolMetadata(
        name="query_tool",
        description="Provides information about the 2023 SF Budget Report.",
    ),
)

In [ ]:
from llama_index.core.agent import FunctionCallingAgentWorker

agent_worker = FunctionCallingAgentWorker.from_tools(
    [kg_query_tool],
    llm=llm,
    verbose=True,
    allow_parallel_tool_calls=False,
)
agent = agent_worker.as_agent()

## Try out Queries

In [ ]:
response = agent.chat("Give me all the programs that the mayor's budget includes")

Added user message to memory: Give me all the programs that the mayor's budget includes
=== Calling Function ===
Calling function: query_tool with args: {"input": "mayor's budget programs"}
=== Function Output ===
The mayor's budget includes a variety of programs and initiatives. It balances departmental budget reductions with investments and is organized into sections that provide citywide budgetary and fiscal policy information. The budget makes significant investments, including funding for a legislative management system, and supports new positions. It also includes allocations for various fiscal years, such as $0.7 million in FY 2023-24 and $0.6 million in FY 2024-25, among others. Programs supported by the budget include the expansion of the First Year Free initiative for small businesses, investments in emergency shelter and supportive housing, and the continuation of the Dream Keeper Initiative to support Black communities. Additionally, the budget advances the Climate Action P

In [ ]:
print(str(response))

The mayor's budget includes a variety of programs and initiatives, such as:

1. **First Year Free Initiative**: Expansion to support small businesses.
2. **Emergency Shelter and Supportive Housing**: Investments to enhance these services.
3. **Dream Keeper Initiative**: Continued support for Black communities.
4. **Climate Action Plan**: Advancements to address climate change.
5. **Legislative Management System**: Funding for implementation.
6. **Hiring and Contracting Reforms**: Improvements in processes.

The budget also includes specific allocations for fiscal years, such as $0.7 million in FY 2023-24 and $0.6 million in FY 2024-25, and supports new positions across various departments.


In [ ]:
agent.reset()
response = agent.chat(
    "Compare the budget for DPA Police Accountabilty from 2022-2023 to 2023-2024"
)
print(str(response))

Added user message to memory: Compare the budget for DPA Police Accountabilty from 2022-2023 to 2023-2024
=== Calling Function ===
Calling function: query_tool with args: {"input": "DPA Police Accountability budget for 2022-2023"}
=== Function Output ===
The budget for the Department of Police Accountability (DPA) for the fiscal year 2022-2023 was $9,776,177.
=== Calling Function ===
Calling function: query_tool with args: {"input": "DPA Police Accountability budget for 2023-2024"}
=== Function Output ===
The budget for the Department of Police Accountability for the fiscal year 2023-2024 is $9.99 million.
=== LLM Response ===
The budget for the Department of Police Accountability (DPA) increased from $9,776,177 in the fiscal year 2022-2023 to $9.99 million in the fiscal year 2023-2024.
The budget for the Department of Police Accountability (DPA) increased from $9,776,177 in the fiscal year 2022-2023 to $9.99 million in the fiscal year 2023-2024.


In [ ]:
print(str(response.source_nodes[0].get_content()))

Here are some facts extracted from the provided text:

Department of police accountability -> Is -> Dpa police accountabilty

SHERIFF ACCOUNTABILITY
                                            MISSION
The Sheriff’s Department of Accountability (SDA), Office of Inspector General (OIG) is
committed to providing the City and County of San Francisco with professional, fair,
and impartial oversight of the San Francisco Sheriff’s Office (SFSO) consistent with
community values and concerns, through thorough investigations, comprehensive policy
reviews and recommendations, and performance audits to ensure compliance with
applicable laws and policies
                             BUDGET ISSUES & DETAILS
The proposed Fiscal Year (FY) 2023-24 budget           a department head, referred to as the Inspector
of $2.3 million for the Sheriff’s Department of        General, in Fiscal Year (FY) 2023-24. The proposed
Accountability is $0.2 million, or 9.9 percent, lower  budget for FY 2022-23 and FY 2023